# Phase 6: weight blending (WiSE-FT) — Cypriot gain vs Standard Greek loss

Fine-tuning Rumble-2-Flash on Cypriot dev went 47.3% → 17.3% WER, but FLEURS Standard Greek went 14.1% → 39.4%
(`docs/PROJECT_LOG.md` section 15). Here we blend the weights, **W = W0 + λ·(W1 − W0)**, with W0 = zero-shot
Rumble (pinned revision) and W1 = our merged LoRA fine-tune, and score every λ on both test beds:
- Cypriot dev: 280 full-length clips, same settings as `configs/eval_rumble-2-flash_lora.yaml`.
- FLEURS `el_gr` test: the same first 200 clips as notebook 09 part B.

Both use the prompt options the LoRA was trained with (`digits: false, filler: true, verbatim: true`), greedy,
batch 1. No training. λ = 0 and λ = 1 are re-measured here so all points share one GPU and dtype.

Script: `scripts/wiseft_sweep.py`. Batch-1 decoding leaves a big GPU mostly idle, so several processes run in
parallel (one per λ). First run (2026-11-25): T4, float32, 2 processes (~5.3 GB RAM each), about 4.5 h for 6 λ values. On a T4 the 2 CPU cores limit the gain from parallel processes (~12%); an A100 (bf16) would be much faster. Results: `docs/PROJECT_LOG.md` section 16.
Needs `MyDrive/cyasr/cyasr_bundle.zip` (data) and `cyasr_code.zip` (current code, incl. the sweep script).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
CY = '/content/drive/MyDrive/cyasr'
FT = f'{CY}/runs/finetune_rumble-2-flash_lora/best'
OUT = f'{CY}/runs_wiseft'
LAMS = [0.0, 0.25, 0.5, 0.625, 0.75, 1.0]   # 0.625 is saved as lam0.62_*

!rm -rf /content/cyasr && mkdir -p /content/cyasr
!unzip -q {CY}/cyasr_bundle.zip -d /content/cyasr
!unzip -q -o {CY}/cyasr_code.zip -d /content/cyasr
%cd /content/cyasr
!ls scripts/wiseft_sweep.py {FT}
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

In [ ]:
# Cache the FLEURS el_gr clips once (same loader and clips as notebook 09 part B)
import csv, io, os, pickle, tarfile, requests, soundfile as sf
FLEURS = 'https://huggingface.co/datasets/google/fleurs/resolve/main/data'
def load_fleurs(lang, n=None):
    tsv = requests.get(f'{FLEURS}/{lang}/test.tsv', timeout=60); tsv.raise_for_status()
    refs = {}
    for row in csv.reader(io.StringIO(tsv.text), delimiter='\t', quoting=csv.QUOTE_NONE):
        refs.setdefault(row[1], (row[0], row[2]))
    clips = []
    with requests.get(f'{FLEURS}/{lang}/audio/test.tar.gz', stream=True, timeout=60) as r:
        r.raise_for_status(); r.raw.decode_content = True
        with tarfile.open(fileobj=r.raw, mode='r|gz') as tar:
            for m in tar:
                name = os.path.basename(m.name)
                if not m.isfile() or name not in refs:
                    continue
                audio, sr = sf.read(io.BytesIO(tar.extractfile(m).read()), dtype='float32'); assert sr == 16000
                uid, ref = refs[name]
                clips.append({'audio_id': name, 'utt_id': uid, 'ref': ref, 'audio': audio, 'duration_s': len(audio) / sr})
                if n and len(clips) >= n:
                    break
    return clips
FL = '/content/fleurs_el_gr_200.pkl'
if not os.path.exists(FL):
    pickle.dump(load_fleurs('el_gr', 200), open(FL, 'wb'))
print(len(pickle.load(open(FL, 'rb'))), 'FLEURS clips cached')

In [ ]:
# Launch one background process per λ, staggered so the CPU-side weight loading doesn't peak at once.
# Lower PARALLEL on a small GPU / low-RAM runtime (L4: 3, T4: 2); λs are then shared out round-robin.
import subprocess, time
PARALLEL = 2   # T4 standard runtime (12 GB RAM, 2 CPUs): 2 processes; more need High-RAM and more CPUs
os.makedirs(OUT, exist_ok=True)
procs = []
for i in range(PARALLEL):
    lams = [str(l) for l in LAMS[i::PARALLEL]]
    if not lams:
        continue
    log = open(f'/content/wiseft_{i}.log', 'w')
    procs.append(subprocess.Popen(['python', 'scripts/wiseft_sweep.py', '--ft', FT, '--fleurs', FL, '--out', OUT,
                                   '--lams', *lams], stdout=log, stderr=subprocess.STDOUT))
    time.sleep(60)
print([p.pid for p in procs])

In [ ]:
# Progress (re-run until all processes have exited)
!tail -n 3 /content/wiseft_*.log
!nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv
print('running:', sum(p.poll() is None for p in procs), 'of', len(procs))

In [ ]:
# Table, Cypriot-form counts, endpoint checks against notebook 09, paired bootstrap, plot, summary.json
import json, numpy as np
from collections import Counter
from cyasr.metrics import error_counts
from cyasr.normalize import normalize
def m(lam, ds): return json.load(open(f'{OUT}/lam{lam:.2f}_{ds}/metrics.json'))
def recs(lam, ds): return [json.loads(l) for l in open(f'{OUT}/lam{lam:.2f}_{ds}/predictions.jsonl', encoding='utf-8')]
print(f"{'λ':>6} | {'CY basic':>8} {'nodiac':>7} {'CER nod':>7} | {'FL basic':>8} {'nodiac':>7} {'CER':>6} | FLEURS τζαι εν και ειναι | min CY/FL")
for lam in LAMS:
    c, f = m(lam, 'cy_dev'), m(lam, 'fleurs_el_gr'); mk = f['markers']; cl, fl = c['levels'], f['levels']
    print(f"{lam:6.3f} | {cl['basic']['wer']:8.1%} {cl['nodiac']['wer']:7.1%} {cl['nodiac']['cer']:7.1%} | "
          f"{fl['basic']['wer']:8.1%} {fl['nodiac']['wer']:7.1%} {fl['basic']['cer']:6.1%} | {mk['τζαι']:4} {mk['εν']:3} {mk['και']:4} {mk['ειναι']:5} | {c['seconds']/60:.0f}/{f['seconds']/60:.0f}")
def mk_counts(texts):
    cc = Counter(w for t in texts for w in normalize(t, 'nodiac').split()); return [cc[w] for w in ['τζαι', 'και', 'εν', 'ειναι']]
print('CY dev τζαι/και/εν/ειναι  ref', mk_counts([r['ref'] for r in recs(1.0, 'cy_dev')]))
for lam in LAMS: print(f'  λ={lam}', mk_counts([r['hyp'] for r in recs(lam, 'cy_dev')]))
FB = f'{CY}/runs_fleurs'   # notebook 09 part B, same clips, same T4 float32
for lam, run in [(1.0, 'rumble-2-flash_lora_el_gr_b'), (0.0, 'rumble-2-flash_m_el_gr_b')]:
    if os.path.exists(f'{FB}/{run}/predictions.jsonl'):
        old = {json.loads(l)['audio_id']: json.loads(l)['hyp'] for l in open(f'{FB}/{run}/predictions.jsonl', encoding='utf-8')}
        print(f'λ={lam:g} FLEURS identical to {run}:', sum(r['hyp'] == old[r['audio_id']] for r in recs(lam, 'fleurs_el_gr')), '/ 200')
def errs(lam, ds):
    rs = recs(lam, ds)
    return (np.array([error_counts(normalize(r['ref'], 'basic'), normalize(r['hyp'], 'basic'))[0] for r in rs]),
            np.array([len(normalize(r['ref'], 'basic').split()) for r in rs]))
def boot(a, b, n=10000, seed=42):
    (ea, nw), (eb, _) = a, b
    idx = np.random.default_rng(seed).integers(0, len(ea), (n, len(ea)))
    d = (eb[idx].sum(1) - ea[idx].sum(1)) / nw[idx].sum(1)
    return (eb.sum() - ea.sum()) / nw.sum(), np.percentile(d, [2.5, 97.5])
for ds in ['cy_dev', 'fleurs_el_gr']:
    for ref in [0.0, 1.0]:
        for lam in LAMS:
            if lam == ref: continue
            d, ci = boot(errs(ref, ds), errs(lam, ds))
            print(f'{ds:13} λ={lam:.3f} vs λ={ref:.0f}: {d:+.1%} [{ci[0]:+.1%}, {ci[1]:+.1%}]')
import matplotlib.pyplot as plt
cy = [m(l, 'cy_dev')['levels']['basic']['wer'] * 100 for l in LAMS]
fl = [m(l, 'fleurs_el_gr')['levels']['basic']['wer'] * 100 for l in LAMS]
fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.plot(fl, cy, 'o-')
for l, x, y in zip(LAMS, fl, cy): ax.annotate(f'λ={l:g}', (x, y), textcoords='offset points', xytext=(6, 4))
ax.set_xlabel('Standard Greek WER, FLEURS el_gr 200 clips (%)'); ax.set_ylabel('Cypriot dev WER, 280 clips (%)')
ax.set_title('Rumble-2-Flash: blending zero-shot and fine-tuned weights', fontsize=10); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(f'{OUT}/wiseft_tradeoff.png', dpi=150); plt.show()
json.dump({str(l): {'cy_dev': m(l, 'cy_dev')['levels'], 'fleurs_el_gr': m(l, 'fleurs_el_gr')['levels'] | {'markers': m(l, 'fleurs_el_gr')['markers']}} for l in LAMS},
          open(f'{OUT}/summary.json', 'w'), ensure_ascii=False, indent=1)

In [ ]:
# Release the GPU when done (Colab bills idle runtimes too)
from google.colab import runtime
runtime.unassign()